In [ ]:
from pathlib import Path
import os
import gc
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision import transforms

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    log_loss,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree

In [ ]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# Device
# ---------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = DEVICE  # keep both names, because older functions may use `device`

# ---------------------------------------------------------------------
# Common constants
# ---------------------------------------------------------------------
LEVEL_ORDER = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]
LEVELS = LEVEL_ORDER

SIDES = ["Left", "Right"]
TARGET_NAMES = ["Normal/Mild", "Moderate", "Severe"]

LEVEL_TO_ID = {level: i for i, level in enumerate(LEVEL_ORDER)}

SIDE_TO_ID = {
    "center": 0,
    "left": 1,
    "right": 2,
    "Left": 1,
    "Right": 2,
}

SERIES_TO_ID = {
    "Sagittal T2/STIR": 0,
    "Sagittal T1": 1,
    "Axial T2": 2,
}

# ---------------------------------------------------------------------
# Image / crop sizes
# ---------------------------------------------------------------------
IMG_SIZE = 256          # localization input size
IMAGE_SIZE = 224       # classification input size
CLS_IMG_SIZE = 224     # same, just for older helper functions
ROI_SIZE = 96          # ROI crop size

NUM_SLICES = 3         # 2.5D input channels
SLICE_OFFSETS = [-1, 0, 1]

# ---------------------------------------------------------------------
# Localization model config
# ---------------------------------------------------------------------
BASE_CHANNELS = 32
DROPOUT = 0.0

# ---------------------------------------------------------------------
# Classification model names
# ---------------------------------------------------------------------
MODEL_NAMES = [
    "resnet34",
    "efficientnet_b0",
    "densenet169",
    "convnext_base",
    "swin_b",
]

# ---------------------------------------------------------------------
# Diagnosis configs
# ---------------------------------------------------------------------
CONDITION_KEYS = [
    "neural_foraminal_narrowing",
    "subarticular_stenosis",
    "spinal_canal_stenosis",
]

CONDITION_CONFIGS = {
    "neural_foraminal_narrowing": {
        "base_condition": "Neural Foraminal Narrowing",
        "series_description": "Sagittal T1",
    },
    "subarticular_stenosis": {
        "base_condition": "Subarticular Stenosis",
        "series_description": "Axial T2",
    },
    "spinal_canal_stenosis": {
        "base_condition": "Spinal Canal Stenosis",
        "series_description": "Sagittal T2/STIR",
    },
}

# ---------------------------------------------------------------------
# Prior / correction params
# ---------------------------------------------------------------------
PRIOR_LAMBDA = 0.5
PRIOR_SIGMA_X = 28
PRIOR_SIGMA_Y = 0
PRIOR_MAP_SIGMA = 8

# ---------------------------------------------------------------------
# Safety net params
# ---------------------------------------------------------------------
SEVERE_THRESHOLD = 0.65
SEVERE_BOOST = 0.10
CLS_IMG_SIZE = 224

MODEL_NAMES = ["resnet34", "efficientnet_b0", "densenet169", "convnext_base", "swin_b"]


In [ ]:
# ------------------------------------------------------------
# Data settings: same as the training workflow
# ------------------------------------------------------------
IMAGE_SIZE = 224
BATCH_SIZE = 64

ROI_CROP_ENABLED = True
ROI_CROP_SIZE = 96
ROI_FALLBACK_TO_FULL_IMAGE = True
X_COLUMN = None
Y_COLUMN = None

USE_2_5D_INPUT = True
SLICE_CONTEXT_OFFSETS = (-1, 0, 1)
SERIES_GROUP_COLUMNS = None
SLICE_ORDER_COLUMN = None

NUM_WORKERS = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 2

# ------------------------------------------------------------
# Model-head settings used by build_metadata_classifier(...)
# ------------------------------------------------------------
PRETRAINED = True
HIDDEN_DIM = 512
DROPOUT = 0.15


In [1]:
# ============================================================
# LABEL DEFINITIONS
# ============================================================

SEVERITY_MAP = {
    "Normal/Mild": 0,
    "Moderate": 1,
    "Severe": 2,
}

TARGET_NAMES = ["Normal/Mild", "Moderate", "Severe"]
N_CLASSES = len(TARGET_NAMES)

LEVELS = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]

SIDE_TO_ID = {
    "center": 0,
    "left": 1,
    "right": 2,
}

SERIES_TO_ID = {
    "Sagittal T2/STIR": 0,
    "Sagittal T1": 1,
    "Axial T2": 2,
}

LEVEL_TO_ID = {level: idx for idx, level in enumerate(LEVELS)}

# CONDITION
CONDITION_KEYS = [
    "neural_foraminal_narrowing",
    "subarticular_stenosis",
    "spinal_canal_stenosis",
]
BASE_CONDITION = "Neural Foraminal Narrowing"
SERIES_DESCRIPTION = "Sagittal T1"

# Only the selected condition is used for mean/std and ensemble evaluation.
MODELS_TO_TRAIN = CONDITION_KEYS


In [ ]:
# ============================================================
# LOAD SUPPORT FUNCTIONS
# ============================================================
# ============================================================
# TRAINING CONFIGURATION
# ============================================================
# This cell controls loss construction, optimizer/scheduler behavior,
# checkpoint selection, and early stopping.

# ----------------------------
# Epochs and early stopping
# ----------------------------
EPOCHS = 30
MIN_EPOCHS = 8
PATIENCE = 5
MIN_DELTA = 1e-4

# ----------------------------
# Loss configuration
# ----------------------------
#   "weighted_cross_entropy"  -> inverse-frequency class weights + medical multipliers
#   "cross_entropy"           -> unweighted cross entropy
LOSS_NAME = "weighted_cross_entropy"

# Small label smoothing helps reduce overconfident validation loss explosion.
LABEL_SMOOTHING = 0.03

# Class imbalance and medical-priority controls.
# class 0 = Normal/Mild, class 1 = Moderate, class 2 = Severe.
MODERATE_WEIGHT_MULTIPLIER = 1.0
SEVERE_WEIGHT_MULTIPLIER = 1.1

# Keep sampler off when using class-weighted loss.
USE_WEIGHTED_SAMPLER = False

# ----------------------------
# Optimizer configuration
# ----------------------------
OPTIMIZER_NAME = "adamw"

# Slightly lower LR because ROI/2.5D model appears to overfit very fast.
LR = 1e-5

# Keep pretrained backbone updates small.
BACKBONE_LR_MULTIPLIER = 0.1

# Stronger weight decay for regularization.
WEIGHT_DECAY = 1e-3

GRAD_CLIP_NORM = 1.0

# ----------------------------
# Scheduler configuration
# ----------------------------
# For ReduceLROnPlateau, SCHEDULER_MONITOR must be one key from val_metrics,
# e.g. "loss", "log_loss", "macro_f1", "balanced_accuracy".
SCHEDULER_NAME = "reduce_lr_on_plateau"

# I would monitor early-stop score or macro_f1 if your training loop supports it.
SCHEDULER_MONITOR = "macro_f1"

SCHEDULER_CONFIG = {
    "mode": "max",
    "factor": 0.5,
    "patience": 2,
}

# ----------------------------
# Checkpoint and early-stopping scoring
# ----------------------------
# Less recall-only pressure. This still rewards severe recall,
# but severe F1 and general class balance matter equally.
CHECKPOINT_SCORE_WEIGHTS = {
    "severe_recall": 0.25,
    "severe_f1": 0.30,
    "pathological_f1": 0.20,
    "macro_f1": 0.15,
    "balanced_accuracy": 0.10,
}

# Early stopping should track broad validation generalization.
EARLY_STOP_SCORE_WEIGHTS = {
    "macro_f1": 0.40,
    "balanced_accuracy": 0.40,
    "pathological_f1": 0.20,
}


In [ ]:
print("Pipeline config loaded.")